# AMAG Replication — Google Colab Training

**Paper**: AMAG: Additive, Multiplicative and Adaptive Graph Neural Network for Forecasting Neuron Activity (NeurIPS 2023)

## Setup Instructions

### Before running this notebook:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro+)
2. **Upload data to Google Drive** (one-time setup):
   - In your Google Drive, create a folder: `MonkeyNeural/dataset/`
   - Upload these 5 files from your local `dataset/` folder:
     - `train_data_affi.npz` (~323 MB)
     - `train_data_affi_2024-03-20_private.npz`
     - `train_data_beignet.npz` (~86 MB)
     - `train_data_beignet_2022-06-01_private.npz`
     - `train_data_beignet_2022-06-02_private.npz`
3. **Upload code to Google Drive** (one-time setup):
   - Zip your entire `MonkeyNeuralForecastingClaude/` project folder
   - Upload the zip to `MonkeyNeural/` in your Drive
   - OR: just upload the `src/` and `replication/` folders

### After training:
- Checkpoints are saved to Google Drive automatically (see cell below)
- Download them via: Files panel (left sidebar) → right-click → Download
- OR: run the download cell at the bottom of this notebook

In [ ]:
# ── STEP 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Verify Drive is mounted
import os
print('Drive mounted. Contents of MonkeyNeural/:',
      os.listdir('/content/drive/MyDrive/MonkeyNeural') if os.path.exists('/content/drive/MyDrive/MonkeyNeural') else 'NOT FOUND — create the folder first')

In [ ]:
# ── STEP 2: Extract project code ────────────────────────────────────────────
# Option A: If you uploaded a zip of the full project to Drive
# !unzip -q /content/drive/MyDrive/MonkeyNeural/MonkeyNeuralForecastingClaude.zip -d /content/

# Option B: Create project structure and copy just the needed folders
import shutil

DRIVE_PROJECT = '/content/drive/MyDrive/MonkeyNeural'
LOCAL_PROJECT = '/content/MonkeyNeuralForecastingClaude'

os.makedirs(LOCAL_PROJECT, exist_ok=True)

# Copy src/ and replication/ from Drive to local (faster execution)
for folder in ['src', 'replication']:
    src_path = os.path.join(DRIVE_PROJECT, folder)
    dst_path = os.path.join(LOCAL_PROJECT, folder)
    if os.path.exists(src_path):
        if os.path.exists(dst_path):
            shutil.rmtree(dst_path)
        shutil.copytree(src_path, dst_path)
        print(f'Copied {folder}/ to {dst_path}')
    else:
        print(f'WARNING: {src_path} not found in Drive')

# Symlink dataset/ to avoid copying 400MB+
dataset_link = os.path.join(LOCAL_PROJECT, 'dataset')
dataset_src  = os.path.join(DRIVE_PROJECT, 'dataset')
if not os.path.exists(dataset_link) and os.path.exists(dataset_src):
    os.symlink(dataset_src, dataset_link)
    print(f'Symlinked dataset/ → {dataset_src}')

print('\nProject structure:')
for item in sorted(os.listdir(LOCAL_PROJECT)):
    print(f'  {item}/')

In [ ]:
# ── STEP 3: Install dependencies ────────────────────────────────────────────
!pip install -q pyyaml einops
# torch, numpy, scipy, scikit-learn, matplotlib are pre-installed in Colab
print('Dependencies ready')

In [ ]:
# ── STEP 4: Add project to Python path ──────────────────────────────────────
import sys
LOCAL_PROJECT = '/content/MonkeyNeuralForecastingClaude'
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── STEP 5: Quick data check ─────────────────────────────────────────────────
import numpy as np
import os

DATASET_DIR = os.path.join(LOCAL_PROJECT, 'dataset')

for fname in ['train_data_affi.npz', 'train_data_beignet.npz']:
    path = os.path.join(DATASET_DIR, fname)
    if os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        key = 'data' if 'data' in d else list(d.keys())[0]
        arr = d[key]
        print(f'{fname}: shape={arr.shape}, dtype={arr.dtype}')
    else:
        print(f'MISSING: {path}')

In [ ]:
# ── STEP 6: Configure output paths (save results to Drive) ───────────────────
import yaml

DRIVE_PROJECT = '/content/drive/MyDrive/MonkeyNeural'
DRIVE_RESULTS = os.path.join(DRIVE_PROJECT, 'training_results')
os.makedirs(DRIVE_RESULTS, exist_ok=True)

REPLICATION_DIR = os.path.join(LOCAL_PROJECT, 'replication', 'amag')

# Load config
config_path = os.path.join(REPLICATION_DIR, 'config.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

print('Config loaded:')
print(f"  lr={config['training']['lr']}, weight_decay={config['training']['weight_decay']}")
print(f"  epochs={config['training']['n_epochs']}, hidden={config['model']['hidden_size']}")
print(f"  batch={config['data']['batch_size']}, patience={config['training']['patience']}")

In [ ]:
# ── STEP 7: Train AMAG ─────────────────────────────────────────────────────
# Choose which monkey to train:
#   'beignet' — 89 channels, ~86MB data, ~30-45 min on T4
#   'affi'    — 239 channels, ~323MB data, ~2-4 hours on T4
#   'both'    — train both sequentially

MONKEY = 'both'  # <── change to 'affi' or 'beignet' as needed

# Checkpoint dirs (local fast storage during training)
CKPT_DIR_AFFI    = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'checkpoints', 'affi')
CKPT_DIR_BEIGNET = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'checkpoints', 'beignet')
os.makedirs(CKPT_DIR_AFFI, exist_ok=True)
os.makedirs(CKPT_DIR_BEIGNET, exist_ok=True)

import json
from src.data.dataset import get_dataloaders
from src.training.trainer import Trainer
from src.evaluation.evaluator import evaluate_model, print_results, save_results
sys.path.insert(0, os.path.join(LOCAL_PROJECT, 'replication', 'amag'))
# Import from local replication module
import importlib.util
spec = importlib.util.spec_from_file_location('amag_model', os.path.join(REPLICATION_DIR, 'model.py'))
amag_model_mod = importlib.util.load_from_spec = spec
from replication.amag.model import build_model

def train_monkey_colab(monkey, config, ckpt_dir):
    print(f'\n{"="*60}')
    print(f'Training AMAG on {monkey.upper()}')
    print(f'Target MSE: {config["experiment"][f"paper_mse_{monkey}"]}')
    print('='*60)

    data_cfg   = config['data']
    model_cfg  = config['model']
    train_cfg  = config['training'].copy()

    train_loader, val_loader, stats = get_dataloaders(
        monkey,
        batch_size=data_cfg['batch_size'],
        val_fraction=data_cfg['val_fraction'],
        seed=data_cfg['seed'],
    )
    print(f'  Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}')

    model = build_model(monkey, model_cfg)
    info = model.get_model_info()
    print(f'  Model: {info["name"]} | {info["n_params_M"]}M params')

    results_dir = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'results')
    os.makedirs(results_dir, exist_ok=True)

    train_cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_cfg['checkpoint_dir'] = ckpt_dir
    train_cfg['log_path'] = os.path.join(results_dir, f'training_log_{monkey}.json')
    train_cfg['norm_stats'] = stats

    trainer = Trainer(model, train_loader, val_loader, train_cfg)
    train_result = trainer.train()

    trainer.load_best_checkpoint()
    val_results = evaluate_model(model, val_loader)
    print_results(val_results, prefix=monkey)

    save_results(val_results, os.path.join(results_dir, f'final_{monkey}_val.json'))

    # ── Copy checkpoint to Google Drive for persistent storage ──
    drive_ckpt_dir = os.path.join(DRIVE_RESULTS, 'checkpoints', monkey)
    os.makedirs(drive_ckpt_dir, exist_ok=True)
    shutil.copy2(os.path.join(ckpt_dir, 'best.pth'), os.path.join(drive_ckpt_dir, 'best.pth'))
    shutil.copy2(train_cfg['log_path'], os.path.join(DRIVE_RESULTS, f'training_log_{monkey}.json'))
    val_res_src = os.path.join(results_dir, f'final_{monkey}_val.json')
    shutil.copy2(val_res_src, os.path.join(DRIVE_RESULTS, f'final_{monkey}_val.json'))
    print(f'  Checkpoint saved to Drive: {drive_ckpt_dir}/best.pth')

    return val_results, train_result['best_val_mse']

import shutil
summary = {}
monkeys_to_train = ['affi', 'beignet'] if MONKEY == 'both' else [MONKEY]
ckpt_dirs = {'affi': CKPT_DIR_AFFI, 'beignet': CKPT_DIR_BEIGNET}

for m in monkeys_to_train:
    results, best_mse = train_monkey_colab(m, config, ckpt_dirs[m])
    paper_mse = config['experiment'][f'paper_mse_{m}']
    summary[m] = {'val_mse': best_mse, 'paper_mse': paper_mse, 'diff': best_mse - paper_mse,
                  'r2': results['r2']['mean']}

print('\n' + '='*60)
print('FINAL SUMMARY vs PAPER')
print('='*60)
for m, s in summary.items():
    diff = s['diff']
    status = 'MATCH' if abs(diff) < 0.002 else ('BETTER' if diff < 0 else f'+{diff/s["paper_mse"]*100:.1f}%')
    print(f'  {m}: val_mse={s["val_mse"]:.5f} | paper={s["paper_mse"]:.5f} | {status}')

with open(os.path.join(DRIVE_RESULTS, 'training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved to: {DRIVE_RESULTS}/training_summary.json')

In [ ]:
# ── STEP 8: Download checkpoints to your local machine ───────────────────────
# Option A: Download via Colab's built-in download
from google.colab import files
import zipfile

# Zip all results for easy download
zip_path = '/content/amag_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(DRIVE_RESULTS):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, DRIVE_RESULTS)
            zf.write(fpath, arcname)

print(f'Created zip: {zip_path}')
print('Size:', os.path.getsize(zip_path) / 1e6, 'MB')
files.download(zip_path)  # triggers browser download

In [ ]:
# ── OPTIONAL: Run ablation study ─────────────────────────────────────────────
# This runs 6 AMAG variants on Beignet to reproduce paper's Table 2 ablation
# Estimated time: ~3-5 hours total on T4 (6 variants × ~30-45 min each)

# Uncomment to run:
# !cd {LOCAL_PROJECT} && python replication/amag/ablation.py --monkey beignet 2>&1

In [ ]:
# ── OPTIONAL: Plot results ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import json

results_dir = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'results')

for monkey in monkeys_to_train:
    log_path = os.path.join(results_dir, f'training_log_{monkey}.json')
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        log = json.load(f)
    history = log['history']

    epochs     = [h['epoch'] for h in history]
    train_mse  = [h['train_mse'] for h in history]
    val_mse    = [h['val_mse'] for h in history]

    plt.figure(figsize=(10, 4))
    plt.plot(epochs, train_mse, label='Train MSE')
    plt.plot(epochs, val_mse,   label='Val MSE')
    paper_target = config['experiment'][f'paper_mse_{monkey}']
    plt.axhline(y=paper_target, color='r', linestyle='--', label=f'Paper target ({paper_target})')
    plt.xlabel('Epoch')
    plt.ylabel('MSE (normalized)')
    plt.title(f'AMAG Training — {monkey.capitalize()}')
    plt.legend()
    plt.tight_layout()
    plot_path = os.path.join(DRIVE_RESULTS, f'training_curve_{monkey}.png')
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'Plot saved: {plot_path}')